# 03 - Features

We add features one group at a time, and each group has to earn its place on **June**. The bar to beat is the depth-2 tree from `01-frame`: PR AUC 0.339, lift 1.78.

Two rules keep this honest:
- **Only information known at pickup.** Every feature comes from the *Allowed information* in `PROBLEM.md`. Duration, speed, fares and anything settled at drop-off never enter.
- **Nothing learns outside May.** Everything that learns from the data (filling missing values, scaling, encoding) sits inside a scikit-learn `Pipeline`, so `fit` only ever sees May.

In [ ]:
import sys
from pathlib import Path

ROOT = Path.cwd().parent
if str(ROOT) not in sys.path:
    sys.path.insert(0, str(ROOT))

import numpy as np
import pandas as pd
from sklearn.compose import ColumnTransformer
from sklearn.impute import SimpleImputer
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import average_precision_score
from sklearn.pipeline import Pipeline, make_pipeline
from sklearn.preprocessing import OneHotEncoder, StandardScaler
from sklearn.tree import DecisionTreeClassifier

from src.data import load_trips

RANDOM_STATE = 42

trips = load_trips()
train = trips[trips["split"] == "train"]
valid = trips[trips["split"] == "valid"]
y_train, y_valid = train["is_crawl"], valid["is_crawl"]

# Building the features

Each feature uses only its own row, never a statistic over many rows, so it cannot leak information between months.

| Group | Features | Why it might help |
|---|---|---|
| distance | `trip_distance` and its logarithm | Short trips crawl most (57% under half a mile). The log lets a linear model see that the first mile matters much more than the tenth |
| time | pickup hour and day of week | Crawling is a daytime, weekday problem |
| zones | pickup zone, drop-off zone, same zone, airport trip | Midtown crawls, airport trips almost never do |
| trip details | `passenger_count`, `VendorID` | Not clear. We test it to see if it adds anything |

`trip_distance` is treated as the planned route length, as `PROBLEM.md` explains. A `passenger_count` of 0 is treated as missing, as decided in `02-clean-and-split`.

In [ ]:
AIRPORT_ZONES = [1, 132, 138]  # Newark, JFK, LaGuardia


def make_features(df):
    pickup = df["tpep_pickup_datetime"]
    return pd.DataFrame({
        "trip_distance": df["trip_distance"],
        "log_distance": np.log1p(df["trip_distance"]),
        "hour": pickup.dt.hour,
        "day_of_week": pickup.dt.day_of_week,
        "PULocationID": df["PULocationID"],
        "DOLocationID": df["DOLocationID"],
        "same_zone": (df["PULocationID"] == df["DOLocationID"]).astype(int),
        "airport_trip": (df["PULocationID"].isin(AIRPORT_ZONES) | df["DOLocationID"].isin(AIRPORT_ZONES)).astype(int),
        "passenger_count": df["passenger_count"].replace(0, np.nan),
        "VendorID": df["VendorID"],
    })


X_train, X_valid = make_features(train), make_features(valid)
X_train.head()

,trip_distance,log_distance,hour,day_of_week,PULocationID,DOLocationID,same_zone,airport_trip,passenger_count,VendorID
0,0.89,0.636577,0,4,151,238,0,0,1.0,2
1,3.45,1.492904,0,4,144,80,0,0,1.0,2
2,12.00,2.564949,0,4,138,49,0,1,1.0,2
3,0.91,0.647103,0,4,234,170,0,0,4.0,2
4,3.30,1.458615,0,4,68,263,0,0,1.0,1


# The pipeline

The numbers and the categories need different treatment:
- **Numbers** (distance, passengers, the 0/1 flags): fill missing values with the May median, then put them on the same scale. A logistic regression needs that to compare its coefficients fairly.
- **Categories** (hour, day, zones, vendor): one column of 0s and 1s per value (*one-hot*). Hour 15 is not "more" than hour 14, it is just different.

Zones with fewer than 50 May trips are grouped into one "infrequent" category. A zone never seen in May goes there too, so a new zone in June or July does not crash the model.

In [ ]:
GROUPS = {
    "distance": {"numeric": ["trip_distance", "log_distance"], "categorical": []},
    "time": {"numeric": [], "categorical": ["hour", "day_of_week"]},
    "zones": {"numeric": ["same_zone", "airport_trip"], "categorical": ["PULocationID", "DOLocationID"]},
    "trip details": {"numeric": ["passenger_count"], "categorical": ["VendorID"]},
}


def preprocessor(groups):
    numeric = [col for g in groups for col in GROUPS[g]["numeric"]]
    categorical = [col for g in groups for col in GROUPS[g]["categorical"]]
    steps = []
    if numeric:
        steps.append(("num", make_pipeline(SimpleImputer(strategy="median", add_indicator=True), StandardScaler()), numeric))
    if categorical:
        steps.append(("cat", OneHotEncoder(handle_unknown="infrequent_if_exist", min_frequency=50), categorical))
    return ColumnTransformer(steps)


def logistic(groups):
    return Pipeline([("prep", preprocessor(groups)), ("model", LogisticRegression(max_iter=2000))])


def tree(groups, depth):
    model = DecisionTreeClassifier(max_depth=depth, min_samples_leaf=50, random_state=RANDOM_STATE)
    return Pipeline([("prep", preprocessor(groups)), ("model", model)])


def june_scores(pipeline):
    """Fit on May, score June: PR AUC and lift."""
    pipeline.fit(X_train, y_train)
    pr_auc = average_precision_score(y_valid, pipeline.predict_proba(X_valid)[:, 1])
    return {"pr_auc": pr_auc, "lift": pr_auc / y_valid.mean()}

# Logistic regression, one group at a time

In [ ]:
steps = [
    ["distance"],
    ["distance", "time"],
    ["distance", "time", "zones"],
    ["distance", "time", "zones", "trip details"],
]

rows = [{"groups": " + ".join(groups), **june_scores(logistic(groups))} for groups in steps]
pd.DataFrame(rows).round(3)

,groups,pr_auc,lift
0,distance,0.445,2.336
1,distance + time,0.495,2.602
2,distance + time + zones,0.632,3.321
3,distance + time + zones + trip details,0.634,3.329


| Step | June PR AUC | Change | Verdict |
|---|---|---|---|
| distance | 0.445 | +0.106 over the baseline | Keep |
| + time | 0.495 | +0.050 | Keep |
| + zones | 0.632 | **+0.137**, the biggest gain | Keep |
| + trip details | 0.634 | +0.002 | **Drop**: too small to matter, and it brings the missing values of the flex trips |

- **Distance alone already beats every baseline** (lift 2.34 against 1.78). The depth-2 tree could only cut distance into four bands; the logistic regression uses it as a smooth curve.
- **Zones add the most**, because they carry what distance and time cannot: *where* in the city the trip starts and ends. This matches the map in `02-clean-and-split`.

# Decision tree, choosing the depth on June

Same features as the best logistic regression: distance, time and zones. Each leaf needs at least 50 May trips, so the tree cannot memorise single trips.

In [ ]:
FINAL_GROUPS = ["distance", "time", "zones"]

rows = [{"max_depth": depth, **june_scores(tree(FINAL_GROUPS, depth))} for depth in [4, 8, 12, 16, 20]]
pd.DataFrame(rows).round(3)

,max_depth,pr_auc,lift
0,4,0.404,2.123
1,8,0.480,2.522
2,12,0.528,2.772
3,16,0.558,2.933
4,20,0.574,3.015


The tree gets better as it gets deeper (PR AUC 0.574 at depth 20, still rising), but it stays well below the logistic regression.

The reason is the zones. With one-hot zones, each split of the tree can only separate *one* zone from all the others, so it needs many levels to learn what the logistic regression gets from one coefficient per zone.

# Conclusion

| Model | Features | June PR AUC | Lift |
|---|---|---|---|
| Baseline: tree, depth 2 | hour, day, distance | 0.339 | 1.78 |
| Decision tree, depth 20 | distance, time, zones | 0.574 | 3.02 |
| **Logistic regression** | **distance, time, zones** | **0.632** | **3.32** |

**The model we take to evaluation is the logistic regression with distance, time and zones.** It almost doubles the baseline's PR AUC on June.